In [59]:
import numpy as np 
from einops import rearrange
from sklearn.preprocessing import StandardScaler, MinMaxScaler


dat = np.stack((np.arange(0,10), np.arange(10,20)))

data = dat.reshape(2,1,5,2)

res = rearrange(data,"batch joint xyz seq -> batch (joint xyz) seq")
shp = res.shape 

scaler = MinMaxScaler()

print(res)

scaler.fit_transform(res.reshape(-1, shp[2])).reshape(shp)



[[[ 0  1]
  [ 2  3]
  [ 4  5]
  [ 6  7]
  [ 8  9]]

 [[10 11]
  [12 13]
  [14 15]
  [16 17]
  [18 19]]]


array([[[0.        , 0.        ],
        [0.11111111, 0.11111111],
        [0.22222222, 0.22222222],
        [0.33333333, 0.33333333],
        [0.44444444, 0.44444444]],

       [[0.55555556, 0.55555556],
        [0.66666667, 0.66666667],
        [0.77777778, 0.77777778],
        [0.88888889, 0.88888889],
        [1.        , 1.        ]]])

In [61]:
import numpy as np
from einops import rearrange
from sklearn.preprocessing import MinMaxScaler

class FeatureScaler:
    def __init__(self):
        self.scaler = MinMaxScaler()
    
    def fit(self, data):
        """
        Fits the scaler on the 3D data.
        
        Parameters:
            data (numpy.ndarray): A 3D numpy array with shape (batch, feature, sequence).
        """
        data_reshaped = rearrange(data, 'b f s -> (b s) f')
        self.scaler.fit(data_reshaped)
    
    def transform(self, data):
        """
        Transforms the data using the already fitted scaler.
        
        Parameters:
            data (numpy.ndarray): A 3D numpy array with shape (batch, feature, sequence).
            
        Returns:
            numpy.ndarray: The normalized data.
        """
        data_reshaped = rearrange(data, 'b f s -> (b s) f')
        normalized_data = self.scaler.transform(data_reshaped)
        normalized_data = rearrange(normalized_data, '(b s) f -> b f s', b=data.shape[0], s=data.shape[2])
        return normalized_data
    
    def fit_transform(self, data):
        """
        Fits the scaler on the data and transforms it.
        
        Parameters:
            data (numpy.ndarray): A 3D numpy array with shape (batch, feature, sequence).
            
        Returns:
            numpy.ndarray: The normalized data.
        """
        self.fit(data)
        return self.transform(data)

# Example usage:
# Example data: 2 batches, 5 features, 2 sequence lengths
data_example = np.stack((np.arange(10), np.arange(10,20))).reshape(2, 5, 2)  # Random data for illustration

# Create an instance of the FeatureScaler class
scaler = FeatureScaler()

# Fit the scaler and transform the data
normalized_data = scaler.fit_transform(data_example)
print(normalized_data)

# You can now use the scaler instance to transform new data with the same parameters
# new_data = ...  # some new data in the same structure as `data_example`
# new_normalized_data = scaler.transform(new_data)



[[[0.         0.09090909]
  [0.         0.09090909]
  [0.         0.09090909]
  [0.         0.09090909]
  [0.         0.09090909]]

 [[0.90909091 1.        ]
  [0.90909091 1.        ]
  [0.90909091 1.        ]
  [0.90909091 1.        ]
  [0.90909091 1.        ]]]


In [46]:
scaler.transform(np.arange(0,10).reshape(1,5,2))

array([[[0.        , 0.09090909],
        [0.        , 0.09090909],
        [0.        , 0.09090909],
        [0.        , 0.09090909],
        [0.        , 0.09090909]]])

In [47]:
import torch
import torch.nn as nn

class FeatureBatchNorm:
    def __init__(self, num_features):
        """
        Initializes the BatchNorm layer for 1D batch normalization.
        
        Parameters:
            num_features (int): Number of features; this corresponds to the number of features
                                in the data assuming data shape is (batch, feature, sequence).
        """
        self.bn = nn.BatchNorm1d(num_features)

    def fit_transform(self, x):
        """
        Applies Batch Normalization to the input data. BatchNorm in training mode inherently fits
        to the data it receives and then transforms it.
        
        Parameters:
            x (torch.Tensor): Input data tensor with shape (batch, feature, sequence).
        
        Returns:
            torch.Tensor: Normalized data tensor.
        """
        # Ensure the model is in training mode for fit_transform
        self.bn.train()
        return self.bn(x)

    def transform(self, x):
        """
        Applies the Batch Normalization transformation using learned parameters. 
        It should only be used after fit_transform or if the parameters are already learned.
        
        Parameters:
            x (torch.Tensor): Input data tensor with shape (batch, feature, sequence).
        
        Returns:
            torch.Tensor: Normalized data tensor.
        """
        # Ensure the model is in evaluation mode for transform, using learned parameters without further training
        self.bn.eval()
        return self.bn(x)

# Example usage:
data_example = torch.randn(2, 5, 2)  # Simulate some data: 2 batches, 5 features, 2 sequence lengths

# Create an instance of FeatureBatchNorm class
batch_norm = FeatureBatchNorm(num_features=5)

# Normalize the data
normalized_data = batch_norm.fit_transform(data_example)
print(normalized_data)

# To transform new data using the same learned parameters (after training):
# new_data_example = torch.randn(2, 5, 2)
# normalized_new_data = batch_norm.transform(new_data_example)
# print(normalized_new_data)


tensor([[[-1.3620,  1.0757],
         [-1.5534,  0.4879],
         [ 1.4638,  0.3779],
         [ 0.0266, -1.0447],
         [ 1.1777,  0.7925]],

        [[-0.5448,  0.8312],
         [ 1.1577, -0.0922],
         [-0.8251, -1.0167],
         [-0.5840,  1.6021],
         [-0.8366, -1.1336]]], grad_fn=<NativeBatchNormBackward0>)


In [65]:
import torch
import torch.nn as nn

class MinMaxScaler(nn.Module):
    def __init__(self, feature_range=(0, 1)):
        super().__init__()
        self.min = feature_range[0]
        self.max = feature_range[1]
        self.data_min = None
        self.data_max = None
        self.scale = None
        self.min_adjust = None

    def fit(self, x):
        self.data_min = torch.min(x, dim=2, keepdim=True)[0]
        self.data_max = torch.max(x, dim=2, keepdim=True)[0]
        self.scale = (self.max - self.min) / (self.data_max - self.data_min)
        self.min_adjust = self.min - self.data_min * self.scale

    def transform(self, x):
        return x * self.scale + self.min_adjust

    def fit_transform(self, x):
        self.fit(x)
        return self.transform(x)
    
scaler = MinMaxScaler()
scaler.fit_transform(torch.from_numpy(data)).float()

tensor([[[[0.0000, 0.0000],
          [0.2500, 0.2500],
          [0.5000, 0.5000],
          [0.7500, 0.7500],
          [1.0000, 1.0000]]],


        [[[0.0000, 0.0000],
          [0.2500, 0.2500],
          [0.5000, 0.5000],
          [0.7500, 0.7500],
          [1.0000, 1.0000]]]])

In [89]:
from sklearn.preprocessing import MinMaxScaler

values = np.arange(0,10).reshape(-1,1)

feat_1 = np.stack((np.arange(0,10), np.arange(20,30))).reshape(20,1)
feat_2 = np.stack((np.arange(500,510), np.arange(700,710))).reshape(20,1)

values = np.concatenate((feat_1, feat_2), axis=1).reshape(2,10,2)



# scaler = MinMaxScaler(feature_range=(0, 1))
# scaler = scaler.fit(values)
# print(f"Min: {scaler.data_min_}, Max: {scaler.data_max_}")
# # normalize the dataset and print the first 5 rows
# normalized = scaler.transform(values).reshape(2,10,2)

# normalized[...,1]


class Scaler:

    def __init__(self) -> None:
        self.__scaler = MinMaxScaler()

    def fit(self, data: np.ndarray) -> None:
        _data = rearrange(data, 'b s f -> (b s) f')
        self.__scaler.fit(_data)

    def transform(self, data: np.ndarray) -> np.ndarray:
        _data = rearrange(data, 'b s f -> (b s) f')
        _normalized = self.__scaler.transform(_data)
        return rearrange(_normalized, '(b s) f -> b s f', b=data.shape[0], s=data.shape[1])
    
    def fit_transform(self, data: np.ndarray) -> np.ndarray:
        self.fit(data)
        return self.transform(data)
    

scaler = Scaler()

values[0,:,1]

scaler.fit_transform(values)[1,:,0]



array([0.68965517, 0.72413793, 0.75862069, 0.79310345, 0.82758621,
       0.86206897, 0.89655172, 0.93103448, 0.96551724, 1.        ])

In [ ]:
import 